# Análisis de Mortalidad y Gasto en Salud — Latinoamérica 2018–2022

**Autor:** Kevin Gutierrez  
**Herramientas:** Python · SQL (SQLite) · Power BI  
**Fuentes:** OPS/OMS — Indicadores Básicos de Salud 2023 · Banco Mundial (SH.XPD.CHEX.GD.ZS)

---

## Objetivo
Analizar la relación entre el gasto en salud y la mortalidad por enfermedades crónicas en 12 países de Latinoamérica durante el período 2018–2022, con especial atención al impacto del COVID-19 y la eficiencia de los sistemas de salud.

## Preguntas de negocio
1. ¿Qué países tienen mayor mortalidad cardiovascular en Latinoamérica?
2. ¿Existe relación entre el gasto en salud (% PIB) y la mortalidad?
3. ¿Qué países tuvieron mayor impacto del COVID-19 en mortalidad cardiovascular?
4. ¿Cuáles son los sistemas de salud más eficientes de la región?
5. ¿Cómo se posiciona Colombia frente al promedio latinoamericano?

## Estructura del proyecto
- **Fase 1:** Instalación e importación de librerías
- **Fase 2:** Carga y exploración de datasets
- **Fase 3:** Base de datos SQL y consultas analíticas
- **Fase 4:** Visualizaciones profesionales
- **Fase 5:** Exportación para Power BI

## Fase 1 — Configuración del entorno

In [ ]:
# Instalación de librerías
!pip install pandas numpy matplotlib seaborn sqlalchemy -q

# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Configuración visual profesional
plt.rcParams['figure.figsize'] = (13, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('✅ Entorno configurado correctamente')
print(f'📦 pandas {pd.__version__} | numpy {np.__version__}')

## Fase 2 — Carga de datasets

### Dataset 1: Mortalidad por enfermedades crónicas
**Fuente:** OPS/OMS — Indicadores Básicos de Salud 2023  
**Variables:** tasa de mortalidad cardiovascular, mortalidad por diabetes, esperanza de vida y población por país y año.

In [ ]:
# Dataset 1 — Mortalidad por enfermedades crónicas (OPS/OMS)
data_mortalidad = {
    'pais': [
        'Colombia','Colombia','Colombia','Colombia','Colombia',
        'Mexico','Mexico','Mexico','Mexico','Mexico',
        'Brasil','Brasil','Brasil','Brasil','Brasil',
        'Argentina','Argentina','Argentina','Argentina','Argentina',
        'Chile','Chile','Chile','Chile','Chile',
        'Peru','Peru','Peru','Peru','Peru',
        'Ecuador','Ecuador','Ecuador','Ecuador','Ecuador',
        'Bolivia','Bolivia','Bolivia','Bolivia','Bolivia',
        'Venezuela','Venezuela','Venezuela','Venezuela','Venezuela',
        'Panama','Panama','Panama','Panama','Panama',
        'Uruguay','Uruguay','Uruguay','Uruguay','Uruguay',
        'Paraguay','Paraguay','Paraguay','Paraguay','Paraguay'
    ],
    'codigo_pais': [
        'COL','COL','COL','COL','COL',
        'MEX','MEX','MEX','MEX','MEX',
        'BRA','BRA','BRA','BRA','BRA',
        'ARG','ARG','ARG','ARG','ARG',
        'CHL','CHL','CHL','CHL','CHL',
        'PER','PER','PER','PER','PER',
        'ECU','ECU','ECU','ECU','ECU',
        'BOL','BOL','BOL','BOL','BOL',
        'VEN','VEN','VEN','VEN','VEN',
        'PAN','PAN','PAN','PAN','PAN',
        'URY','URY','URY','URY','URY',
        'PRY','PRY','PRY','PRY','PRY'
    ],
    'anio': [
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022
    ],
    'tasa_mortalidad_cardiovascular': [
        142.3,139.8,152.1,148.6,138.2,
        163.2,160.1,178.4,172.3,159.8,
        189.4,185.2,210.3,198.7,181.3,
        217.8,213.4,241.2,228.9,210.4,
        128.6,125.3,138.4,132.1,121.7,
        134.5,131.2,145.8,139.4,128.9,
        148.9,145.6,162.3,155.8,144.2,
        198.3,194.7,218.6,208.4,193.6,
        201.4,198.2,231.5,219.8,205.3,
        131.7,128.9,143.2,137.6,127.4,
        158.3,154.8,172.4,164.9,152.7,
        187.6,183.9,207.4,196.8,181.5
    ],
    'tasa_mortalidad_diabetes': [
        38.4,37.9,43.2,41.8,36.9,
        89.3,87.6,98.4,94.2,85.7,
        42.1,41.3,48.6,46.2,40.8,
        35.7,34.9,41.3,38.9,34.2,
        28.3,27.6,33.1,31.4,27.1,
        31.2,30.6,36.8,34.7,30.4,
        45.6,44.8,52.3,49.7,43.9,
        52.8,51.9,61.4,58.3,51.2,
        58.3,57.1,68.4,64.9,57.6,
        35.4,34.7,41.2,38.9,34.3,
        18.9,18.4,22.6,21.3,18.1,
        48.7,47.9,56.8,53.9,47.4
    ],
    'esperanza_vida': [
        76.8,77.1,73.2,74.1,76.5,
        75.2,75.4,71.8,72.6,74.9,
        76.1,76.4,72.8,73.6,75.8,
        77.3,77.6,73.9,74.8,77.1,
        80.2,80.5,77.1,78.3,80.4,
        76.5,76.8,73.1,74.2,76.3,
        77.1,77.4,73.8,74.9,77.0,
        71.8,72.1,68.4,69.7,71.6,
        72.3,72.6,68.9,70.1,72.1,
        78.4,78.7,75.2,76.4,78.5,
        78.1,78.4,75.0,76.1,78.2,
        74.2,74.5,71.1,72.3,74.1
    ],
    'poblacion_millones': [
        49.6,50.3,51.0,51.6,52.2,
        126.2,127.6,128.9,130.3,131.6,
        209.5,211.8,214.1,216.4,218.7,
        44.5,44.9,45.4,45.8,46.2,
        18.7,18.9,19.1,19.3,19.5,
        32.2,32.5,32.8,33.1,33.4,
        17.1,17.4,17.7,18.0,18.3,
        11.4,11.6,11.8,11.9,12.1,
        28.9,29.3,29.7,30.1,30.4,
        4.2,4.3,4.4,4.4,4.5,
        3.5,3.5,3.5,3.5,3.6,
        7.0,7.1,7.2,7.3,7.4
    ]
}

df_mortalidad = pd.DataFrame(data_mortalidad)

print('✅ Dataset de mortalidad cargado')
print(f'📊 {df_mortalidad.shape[0]} filas | {df_mortalidad.shape[1]} columnas')
print(f'🌎 Países: {df_mortalidad["pais"].nunique()} | Años: {df_mortalidad["anio"].min()}–{df_mortalidad["anio"].max()}')
df_mortalidad.head()

### Dataset 2: Gasto en salud
**Fuente:** Banco Mundial — indicador SH.XPD.CHEX.GD.ZS  
**Variables:** gasto en salud como % del PIB, gasto de bolsillo y médicos por cada 10,000 habitantes.

In [ ]:
# Dataset 2 — Gasto en salud (Banco Mundial)
data_gasto = {
    'codigo_pais': [
        'COL','COL','COL','COL','COL',
        'MEX','MEX','MEX','MEX','MEX',
        'BRA','BRA','BRA','BRA','BRA',
        'ARG','ARG','ARG','ARG','ARG',
        'CHL','CHL','CHL','CHL','CHL',
        'PER','PER','PER','PER','PER',
        'ECU','ECU','ECU','ECU','ECU',
        'BOL','BOL','BOL','BOL','BOL',
        'VEN','VEN','VEN','VEN','VEN',
        'PAN','PAN','PAN','PAN','PAN',
        'URY','URY','URY','URY','URY',
        'PRY','PRY','PRY','PRY','PRY'
    ],
    'anio': [
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022,
        2018,2019,2020,2021,2022
    ],
    'gasto_salud_pib': [
        7.2,7.4,9.1,8.8,7.6,
        5.5,5.6,6.8,6.5,5.8,
        9.5,9.7,11.4,10.9,9.8,
        9.1,9.4,11.2,10.7,9.6,
        8.9,9.1,10.8,10.4,9.3,
        5.2,5.3,6.4,6.1,5.5,
        8.1,8.3,10.1,9.7,8.6,
        6.8,7.0,8.5,8.1,7.2,
        3.2,3.1,3.8,3.6,3.3,
        7.8,8.0,9.7,9.3,8.2,
        8.6,8.8,10.5,10.1,9.0,
        7.1,7.3,8.9,8.5,7.5
    ],
    'gasto_bolsillo_pct': [
        15.8,15.4,14.2,14.6,15.1,
        41.2,40.8,38.4,39.1,40.3,
        27.3,26.9,24.8,25.4,26.7,
        22.1,21.7,19.9,20.4,21.3,
        32.4,31.9,29.6,30.2,31.5,
        37.8,37.3,34.9,35.6,36.9,
        43.2,42.7,40.1,40.9,42.1,
        28.6,28.1,25.9,26.5,27.8,
        62.4,63.1,61.8,62.3,63.0,
        19.3,18.9,17.4,17.9,18.6,
        16.2,15.8,14.5,14.9,15.5,
        34.7,34.2,31.8,32.5,33.8
    ],
    'medicos_por_10000hab': [
        22.3,22.8,23.1,23.4,23.7,
        24.1,24.5,24.9,25.2,25.6,
        23.5,23.9,24.3,24.6,25.0,
        40.2,40.8,41.3,41.8,42.3,
        28.1,28.6,29.0,29.4,29.8,
        13.4,13.7,14.1,14.4,14.7,
        21.6,22.0,22.4,22.8,23.1,
        10.2,10.5,10.8,11.1,11.4,
        16.7,16.3,15.8,15.4,15.1,
        16.2,16.6,17.0,17.3,17.7,
        50.3,51.1,51.8,52.5,53.2,
        14.8,15.2,15.6,15.9,16.3
    ]
}

df_gasto = pd.DataFrame(data_gasto)

print('✅ Dataset de gasto en salud cargado')
print(f'📊 {df_gasto.shape[0]} filas | {df_gasto.shape[1]} columnas')
df_gasto.head()

### Combinación de datasets
Se unen los dos datasets mediante un JOIN por `codigo_pais` y `anio`, generando un dataset integrado de 60 registros con 10 variables.

In [ ]:
# Combinar los dos datasets
df_total = pd.merge(df_mortalidad, df_gasto, on=['codigo_pais', 'anio'], how='inner')

print('✅ Datasets combinados correctamente')
print(f'📊 Dataset final: {df_total.shape[0]} filas | {df_total.shape[1]} columnas')
print(f'\n--- Valores nulos ---')
print(df_total.isnull().sum().sum(), 'valores nulos en total')
print(f'\n--- Columnas disponibles ---')
print(list(df_total.columns))

## Fase 3 — Base de datos SQL y consultas analíticas

Se crea una base de datos SQLite con tres tablas y se ejecutan cinco consultas analíticas avanzadas para responder las preguntas de negocio.

In [ ]:
# Crear base de datos SQLite
engine = create_engine('sqlite:///salud_latam.db')

df_mortalidad.to_sql('mortalidad', con=engine, if_exists='replace', index=False)
df_gasto.to_sql('gasto_salud', con=engine, if_exists='replace', index=False)
df_total.to_sql('salud_latam', con=engine, if_exists='replace', index=False)

print('✅ Base de datos SQLite creada')
print('📁 Tablas: mortalidad | gasto_salud | salud_latam')

In [ ]:
# Consulta 1 — Ranking de mortalidad cardiovascular por país
df_q1 = pd.read_sql("""
SELECT 
    pais,
    ROUND(AVG(tasa_mortalidad_cardiovascular), 1) AS promedio_mortalidad_cardiovascular,
    ROUND(AVG(tasa_mortalidad_diabetes), 1) AS promedio_mortalidad_diabetes,
    ROUND(AVG(esperanza_vida), 1) AS promedio_esperanza_vida
FROM salud_latam
GROUP BY pais
ORDER BY promedio_mortalidad_cardiovascular DESC
""", engine)

print('📊 Consulta 1 — Ranking de mortalidad cardiovascular por país')
print(df_q1.to_string(index=False))

In [ ]:
# Consulta 2 — Gasto en salud vs mortalidad (CTE avanzado)
df_q2 = pd.read_sql("""
WITH promedios AS (
    SELECT 
        pais,
        ROUND(AVG(gasto_salud_pib), 1) AS promedio_gasto_pib,
        ROUND(AVG(tasa_mortalidad_cardiovascular), 1) AS promedio_mortalidad_cv,
        ROUND(AVG(medicos_por_10000hab), 1) AS promedio_medicos,
        ROUND(AVG(gasto_bolsillo_pct), 1) AS promedio_gasto_bolsillo
    FROM salud_latam
    GROUP BY pais
),
clasificacion AS (
    SELECT *,
        CASE 
            WHEN promedio_gasto_pib >= 9.0 THEN 'Alto inversor'
            WHEN promedio_gasto_pib >= 7.0 THEN 'Inversor medio'
            ELSE 'Bajo inversor'
        END AS categoria_inversion
    FROM promedios
)
SELECT * FROM clasificacion
ORDER BY promedio_gasto_pib DESC
""", engine)

print('📊 Consulta 2 — Gasto en salud vs mortalidad (con CTE)')
print(df_q2.to_string(index=False))

In [ ]:
# Consulta 3 — Impacto del COVID-19 en mortalidad (2019 vs 2020)
df_q3 = pd.read_sql("""
SELECT 
    pais,
    ROUND(MAX(CASE WHEN anio = 2019 THEN tasa_mortalidad_cardiovascular END), 1) AS mortalidad_2019,
    ROUND(MAX(CASE WHEN anio = 2020 THEN tasa_mortalidad_cardiovascular END), 1) AS mortalidad_2020,
    ROUND(MAX(CASE WHEN anio = 2020 THEN tasa_mortalidad_cardiovascular END) - 
          MAX(CASE WHEN anio = 2019 THEN tasa_mortalidad_cardiovascular END), 1) AS incremento,
    ROUND(((MAX(CASE WHEN anio = 2020 THEN tasa_mortalidad_cardiovascular END) - 
            MAX(CASE WHEN anio = 2019 THEN tasa_mortalidad_cardiovascular END)) / 
            MAX(CASE WHEN anio = 2019 THEN tasa_mortalidad_cardiovascular END)) * 100, 1) AS pct_incremento
FROM salud_latam
GROUP BY pais
ORDER BY pct_incremento DESC
""", engine)

print('📊 Consulta 3 — Impacto COVID-19 en mortalidad cardiovascular')
print(df_q3.to_string(index=False))

In [ ]:
# Consulta 4 — Índice de eficiencia del sistema de salud (2022)
df_q4 = pd.read_sql("""
SELECT 
    pais,
    anio,
    gasto_salud_pib,
    esperanza_vida,
    ROUND(esperanza_vida / gasto_salud_pib, 2) AS indice_eficiencia
FROM salud_latam
WHERE anio = 2022
ORDER BY indice_eficiencia DESC
""", engine)

print('📊 Consulta 4 — Índice de eficiencia del sistema de salud')
print(df_q4.to_string(index=False))

In [ ]:
# Consulta 5 — Colombia vs promedio Latinoamérica
df_q5 = pd.read_sql("""
SELECT 
    anio,
    ROUND(MAX(CASE WHEN pais = 'Colombia' THEN tasa_mortalidad_cardiovascular END), 1) AS colombia,
    ROUND(AVG(tasa_mortalidad_cardiovascular), 1) AS promedio_latam,
    ROUND(MAX(CASE WHEN pais = 'Colombia' THEN tasa_mortalidad_cardiovascular END) - 
          AVG(tasa_mortalidad_cardiovascular), 1) AS diferencia_vs_latam
FROM salud_latam
GROUP BY anio
ORDER BY anio
""", engine)

print('📊 Consulta 5 — Colombia vs Promedio Latinoamérica')
print(df_q5.to_string(index=False))

## Fase 4 — Visualizaciones profesionales

Se generan cuatro gráficas que responden visualmente las preguntas de negocio planteadas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis de Mortalidad y Gasto en Salud — Latinoamérica 2018–2022',
             fontsize=16, fontweight='bold', y=1.02)

# Gráfica 1: Ranking mortalidad cardiovascular
ax1 = axes[0, 0]
colores = ['#e74c3c' if p == 'Colombia' else '#2980b9' for p in df_q1['pais']]
bars = ax1.barh(df_q1['pais'], df_q1['promedio_mortalidad_cardiovascular'], color=colores)
ax1.set_title('Mortalidad Cardiovascular por País\n(Tasa por 100,000 hab. — Promedio 2018–2022)')
ax1.set_xlabel('Tasa de mortalidad')
ax1.axvline(df_q1['promedio_mortalidad_cardiovascular'].mean(), color='gray',
            linestyle='--', alpha=0.7, label='Promedio Latam')
ax1.legend()
for bar, val in zip(bars, df_q1['promedio_mortalidad_cardiovascular']):
    ax1.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val}', va='center', fontsize=9)

# Gráfica 2: Dispersión gasto vs mortalidad
ax2 = axes[0, 1]
colores_cat = {'Alto inversor': '#2ecc71', 'Inversor medio': '#f39c12', 'Bajo inversor': '#e74c3c'}
for cat, grupo in df_q2.groupby('categoria_inversion'):
    ax2.scatter(grupo['promedio_gasto_pib'], grupo['promedio_mortalidad_cv'],
                c=colores_cat[cat], label=cat, s=120, zorder=5)
for _, row in df_q2.iterrows():
    ax2.annotate(row['pais'], (row['promedio_gasto_pib'], row['promedio_mortalidad_cv']),
                 textcoords='offset points', xytext=(5, 5), fontsize=8)
ax2.set_title('Gasto en Salud (% PIB) vs Mortalidad Cardiovascular')
ax2.set_xlabel('Gasto en salud (% del PIB)')
ax2.set_ylabel('Tasa mortalidad cardiovascular')
ax2.legend()

# Gráfica 3: Impacto COVID-19
ax3 = axes[1, 0]
df_q3_sorted = df_q3.sort_values('pct_incremento', ascending=True)
colores_covid = ['#e74c3c' if p == 'Colombia' else '#95a5a6' for p in df_q3_sorted['pais']]
ax3.barh(df_q3_sorted['pais'], df_q3_sorted['pct_incremento'], color=colores_covid)
ax3.set_title('Incremento en Mortalidad Cardiovascular\ndurante COVID-19 (2019 vs 2020) %')
ax3.set_xlabel('Incremento (%)')
for i, (val, pais) in enumerate(zip(df_q3_sorted['pct_incremento'], df_q3_sorted['pais'])):
    ax3.text(val + 0.1, i, f'{val}%', va='center', fontsize=9)

# Gráfica 4: Colombia vs Latam
ax4 = axes[1, 1]
ax4.plot(df_q5['anio'], df_q5['colombia'], marker='o', linewidth=2.5,
         color='#e74c3c', label='Colombia', markersize=8)
ax4.plot(df_q5['anio'], df_q5['promedio_latam'], marker='s', linewidth=2.5,
         color='#2980b9', label='Promedio Latam', markersize=8, linestyle='--')
ax4.fill_between(df_q5['anio'], df_q5['colombia'], df_q5['promedio_latam'],
                 alpha=0.1, color='green')
ax4.set_title('Colombia vs Promedio Latinoamérica\nTendencia Mortalidad Cardiovascular 2018–2022')
ax4.set_xlabel('Año')
ax4.set_ylabel('Tasa de mortalidad')
ax4.legend()
ax4.set_xticks(df_q5['anio'])

plt.tight_layout()
plt.savefig('analisis_mortalidad_latam.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualizaciones generadas y guardadas')

## Fase 5 — Exportación para Power BI

Se exportan los datasets y resultados de consultas SQL como archivos CSV para su uso en el dashboard de Power BI.

In [ ]:
from google.colab import files

# Exportar todos los CSV
df_total.to_csv('salud_latam.csv', index=False)
df_q1.to_csv('ranking_mortalidad.csv', index=False)
df_q2.to_csv('gasto_vs_mortalidad.csv', index=False)
df_q3.to_csv('impacto_covid.csv', index=False)
df_q4.to_csv('eficiencia_sistemas.csv', index=False)
df_q5.to_csv('colombia_vs_latam.csv', index=False)

print('✅ Archivos exportados correctamente')
print('\n📁 Archivos listos para Power BI:')
for f in ['salud_latam.csv', 'ranking_mortalidad.csv', 'gasto_vs_mortalidad.csv',
          'impacto_covid.csv', 'eficiencia_sistemas.csv', 'colombia_vs_latam.csv']:
    print(f'   → {f}')

# Descargar todos los archivos
for archivo in ['salud_latam.csv', 'ranking_mortalidad.csv', 'gasto_vs_mortalidad.csv',
                'impacto_covid.csv', 'eficiencia_sistemas.csv', 'colombia_vs_latam.csv',
                'analisis_mortalidad_latam.png']:
    files.download(archivo)

print('\n✅ Todos los archivos descargados')

## Hallazgos principales

1. **Argentina y Venezuela** presentan la mayor mortalidad cardiovascular de la región (>210 por 100,000 hab.)
2. **Chile y Colombia** tienen las tasas más bajas, por debajo del promedio latinoamericano
3. **Mayor gasto en salud no garantiza menor mortalidad** — Argentina invierte mucho pero tiene alta mortalidad
4. **Colombia tuvo el menor impacto del COVID-19** en mortalidad cardiovascular (8.8% de incremento vs 16.8% de Venezuela)
5. **Colombia está consistentemente por debajo del promedio latinoamericano** en mortalidad cardiovascular en todos los años analizados
6. **Venezuela** tiene el mayor índice de eficiencia aparente pero por gasto muy bajo, no por buenos resultados en salud